# Filtros de Fase Linear usando Janelamento

## Bibliotecas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import scipy.signal as signal

## Parte 1

1. Gere um sinal de entrada $x[n]$ que é a soma de três funções cosseno com frequências angulares iguais a 0,2π, 0,5π e 0,8π, e amplitudes iguais a 1. Este sinal deve possuir N=200 pontos. Gere o gráfico do módulo da Transformada de Fourier deste sinal $x[n]$ em dB. Não esqueça de colocar as frequências corretas no eixo x. Explique, em forma de comentário em seu código, se a módulo da Transformada de Fourier está de acordo com o esperado.

Gerando o sinal $ x[n] $ formado por cossenos com as seguintes frequências: $ 0,2\pi $, $ 0,5\pi $ e $ 0,8\pi $ com aplitude igual a 1

In [ ]:
x_N = 200

x_n = np.arange(x_N)
x = np.cos(0.2 * np.pi * x_n) + \
    np.cos(0.5 * np.pi * x_n) + \
    np.cos(0.8 * np.pi * x_n)

Calculando a Transformada de Fourier do sinal $ x[n] $ 

In [ ]:
X        = np.fft.fft(x)
X_abs    = np.abs(X)
X_abs_dB = 20 * np.log10(X_abs)

x_freqs            = np.fft.fftfreq(x_N, d = 1)
x_freqs            = np.fft.fftshift(x_freqs)
x_freqs_normalized = 2 * np.pi * x_freqs

Representação gráfica do sinal no tempo e na frequência

In [ ]:
figure, axes = plt.subplots(1, 2, figsize = (16, 6))

axes[0].plot(x_n, x)
axes[0].set_title("$ x[n] $")
axes[0].set_xlabel("Amostra")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True)

axes[1].plot(x_freqs_normalized, X_abs_dB)
axes[1].set_title("$ |X(j\\omega)| $")
axes[1].set_xlabel("Frequência (rad/amostra)")
axes[1].set_ylabel("Amplitude (dB)")
axes[1].grid(True)

plt.show()
plt.close()


Esse gráfico gerado está coerente com o esperado. Visto que a TF do $ cos(\omega x)$ é $ \pi(\delta(\omega - \omega_0) + \delta(\omega - \omega_0))$, temos 6 impulsos gerados pelos 3 cossenos, localizados em suas respectivas frequêcias, que forma o sinal $ x[n] $ 

2. Gere a resposta ao impulso $h[n]$ de um filtro passa-baixa FIR usando truncamento (janela retangular) a partir um filtro passa-baixa ideal com frequência de corte igual a 0,65π e atraso α=15. A resposta ao impulso deste filtro deve ser não nula de 0 até M=2α=30. Gere o gráfico do módulo da Resposta em Frequência deste sistema em dB (ou seja, da Transformada de Fourier da resposta ao impulso $h[n]$). Não esqueça de colocar as frequências corretas no eixo x. Explique, em forma de comentário em seu código, se a módulo da Resposta em Frequência está de acordo com o esperado.

Gerando a resposta ao impulso $h[n]$ de um filtro passa-baixa FIR usando uma janela retangular:

In [ ]:
alpha = 15
M     = 2 * alpha
n     = np.arange(M + 1)
wc    = 0.65 * np.pi

In [ ]:
h = np.zeros_like(n, dtype = float)

for i, k in enumerate(n):
    if k == alpha:
        h[i] = wc / np.pi
    else:
        h[i] = np.sin(wc * (k - alpha)) / (np.pi * (k - alpha))

Calculando a resposta em frequência do filtro

In [ ]:
N_h_fft = 2048

H = np.fft.fft(h, N_h_fft)
H = np.fft.fftshift(H)

H_abs    = np.abs(H)
H_abs_dB = 20 * np.log10(H_abs)

H_frequencies = np.linspace(-np.pi, np.pi, N_h_fft)

Representação gráfica do filtro no tempo e na frequência

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(n, h)
axes[0].set_title("$h[n]$")
axes[0].set_xlabel("n")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True)

axes[1].plot(H_frequencies, H_abs_dB)
axes[1].set_title("Módulo de $H(j\\omega)$ (dB)")
axes[1].set_xlabel("Frequência (rad/amostra)")
axes[1].set_ylabel("Magnitude (dB)")
axes[1].axhline(y = -3, color = 'red', linestyle='--')
axes[1].grid(True)

plt.tight_layout()
plt.show()

Como sabemos, um $sinc$ no tempo é uma janela retangular na na frequência. Perceba que na frequência que acontece a quedra de 3dB, é onde a frequência de corte $\omega_c$ é estipulada. Além disso, como estamos usando uma janela retangular, há lóbulos secundários relevantes na faixa de rejeitção. Dessa forma, o gráfico está coerente com o esperado. 

3. Gere o gráfico da resposta em fase deste sistema (com fase contínua, usando unwrap. Não esqueça de colocar as frequências corretas no eixo x. Explique, em forma de comentário em seu código, se a fase da Resposta em Frequência está de acordo com o esperado.

Calculando a resposta em fase do sistema $h$

In [ ]:
H_phase        = np.angle(H)
H_phase_unwrap = np.unwrap(H_phase)

In [ ]:
figure, axes = plt.subplots(nrows = 1, ncols = 2, figsize = (16, 6))

axes[0].plot(H_frequencies, H_phase)
axes[0].set_title("Fase de $ H(j\\omega) $")
axes[0].set_xlabel("Frequência (rad/amostra)")
axes[0].set_ylabel("Defasamento")

axes[1].plot(H_frequencies, H_phase_unwrap)
axes[1].set_title("Fase $ H(j\\omega) $ (Unwrap)")
axes[1].set_xlabel("Frequência (rad/amostra)")
axes[1].set_ylabel("Defasamento")

plt.show()
plt.close()

Observamos uma fase linear no intervalo do $ [-\omega_c; \omega_c] $ com declividade $ \alpha = 15 $. Portanto, o gráfico satisfaz nossas espectativas

4. Gere o gráfico do atraso de grupo deste sistema (pode usar a função pronta para o cálculo do atraso de grupo). Não esqueça de colocar as frequências corretas no eixo x. Explique, em forma de comentário em seu código, se o atraso de grupo está de acordo com o esperado.

Calculando o atraso de grupo numericamente

In [ ]:
d_H_phase_unwrap = np.diff(H_phase_unwrap)
d_H_frequencies  = np.diff(H_frequencies)

H_group_delay = - d_H_phase_unwrap / d_H_frequencies

In [ ]:
figure, axes = plt.subplots(nrows = 1, ncols = 2, figsize = (16, 6))

axes[0].plot(H_frequencies, H_phase_unwrap)
axes[0].set_title("Fase $ H(j\\omega) $ (Unwrap)")
axes[0].set_xlabel("Frequência (rad/amostra)")
axes[0].set_ylabel("Defasamento")

axes[1].plot(H_frequencies[1:], H_group_delay)
axes[1].set_title("Atraso de Grupo de $ H(j\\omega) $")
axes[1].set_xlabel("Frequência (rad/amostra)")
axes[1].set_ylabel("Defasamento")
axes[1].set_ylim(14, 16)

plt.show()
plt.close()

O atraso de grupo esperado é uma reta horizontal em $ y = 15 $ para a faixa $ [-\omega_c; \omega_c] $. Dessa forma, está como esperado

5. Filtre o sinal x[n] da Questão 1 usando a resposta ao impulso h[n] gerada na Questão 2. Gere o gráfico do módulo da Transformada de Fourier da saída y[n] em dB. Não esqueça de colocar as frequências corretas no eixo x. Explique, em forma de comentário em seu código, se o módulo da Transformada de Fourier do sinal filtrado está de acordo com o esperado.

Realizando a convolução de $ y[n] = x[n] \ast h[n] $

In [ ]:
y   = np.convolve(x, h)
y_n = np.arange(y.size)

Calculando a Transformada de Fourier de $ y[n] $

In [ ]:
N_y_fft = 2048

Y = np.fft.fft(y, N_y_fft)
Y = np.fft.fftshift(Y)

Y_abs    = np.abs(Y)
Y_abs_dB = 20 * np.log10(Y_abs)

Y_frequencies = np.linspace(-np.pi, np.pi, N_y_fft)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(y_n, y)
axes[0].set_title("$y[n]$")
axes[0].set_xlabel("n")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True)

axes[1].plot(Y_frequencies, Y_abs_dB)
axes[1].set_title("Módulo de $Y(j\\omega)$ (dB)")
axes[1].set_xlabel("Frequência (rad/amostra)")
axes[1].set_ylabel("Magnitude (dB)")
axes[1].axhline(y = -3, color = 'red', linestyle='--')
axes[1].grid(True)

plt.tight_layout()
plt.show()

Realizando a convolução de $ y[n] = x[n] \ast h[n] $

In [ ]:
y   = np.convolve(x, h)
y_n = np.arange(y.size)

Calculando a Transformada de Fourier de $ y[n] $

In [ ]:
N_y_fft = 2048

Y = np.fft.fft(y, N_y_fft)
Y = np.fft.fftshift(Y)

Y_abs    = np.abs(Y)
Y_abs_dB = 20 * np.log10(Y_abs)

Y_frequencies = np.linspace(-np.pi, np.pi, N_y_fft)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(y_n, y)
axes[0].set_title("$y[n]$")
axes[0].set_xlabel("n")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True)

axes[1].plot(Y_frequencies, Y_abs_dB)
axes[1].set_title("Módulo de $Y(j\\omega)$ (dB)")
axes[1].set_xlabel("Frequência (rad/amostra)")
axes[1].set_ylabel("Magnitude (dB)")
axes[1].axhline(y = -3, color = 'red', linestyle='--')
axes[1].grid(True)

plt.tight_layout()
plt.show()

Como podemos notar, a componente $\cos(0.8\pi x)$ foi atenuada pelo filtro. Isso é esperado já que estamos utilizando um filtro passa-baixa com frequência de corte $ \omega_c = 0.65\pi $. Daí, componentes com frequências acima de $ \omega_c $ são atenuadas

6. O sinal de saída y[n] deve ser, de forma aproximada, igual a um sinal g[n] que corresponde à soma de dois cossenos com frequências angulares igual a 0,2π e 0,5π, mas com um atraso igual a α. Para saber se isto realmente está acontecendo, gere, em um mesmo gráfico, os sinais g[n-α] e y[n], e comente se estes sinais são parecidos e sincronizados.

Gerando o sinal $ g[n] $

In [ ]:
g = np.cos(0.2 * np.pi * (y_n - alpha)) + \
    np.cos(0.5 * np.pi * (y_n - alpha))

Gerando o sinal $ e[n] = g[n] - y[n] $

In [ ]:
e = g - y

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey = True)

axes[0].plot(y_n, y, alpha = 0.75)
axes[0].plot(y_n, g, alpha = 0.75)
axes[0].set_title("$ g[n] $ e $ y[n] $")
axes[0].set_xlabel("n")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True)

axes[1].plot(y_n, e)
axes[1].set_title("$ e[n] = g[n] - y[n] $")
axes[1].set_xlabel("n")
axes[1].set_ylabel("Amplitude")
axes[1].grid(True)

plt.tight_layout()
plt.show()

Sim. Os gráficos são parecidos e sincronizados. Exceto nas bordas

## Parte 2

7. Gere os coeficientes $a_k$ e $b_k$ da equação de diferenças de um filtro passa-baixa IIR de Butterworth com frequência de corte igual a 0,65π e ordem igual a 8. Use uma função pronta para gerar o filtro de Butterworth. Gere o gráfico do módulo da Resposta em Frequência deste sistema em dB. Não esqueça de colocar as frequências corretas no eixo x. Explique, em forma de comentário em seu código, se a módulo da Resposta em Frequência está de acordo com o esperado.

Parameros do filtro passa-baixa IRR Butterworth

In [ ]:
wc            = 0.65 * np.pi
wc_normalized = wc / np.pi

order      = 8
filterType = 'lowpass'

Coeficientes $a_k$ e $b_k$

In [ ]:
b_k, a_k = signal.butter(
    N      = order, 
    Wn     = wc_normalized, 
    btype  = filterType, 
)

print("b_k =", b_k)
print("a_k =", a_k)

Calculando a resposta em frequência do filtro

In [ ]:
H_frequencies, H = signal.freqz(b_k, a_k, worN = 2048)

H_abs = np.abs(H)
H_dB  = 20 * np.log10(H_abs)

In [ ]:
plt.figure(figsize = (8, 6))

plt.title("Resposta em Frequência do Filtro Butterworth (Ordem 8)")

plt.plot(H_frequencies, H_dB) 

plt.xlabel("Frequência (rad/amostra)")
plt.ylabel("Magnitude (dB)")
plt.axhline(y = -3, color = 'red', linestyle='--')

plt.grid(True)

plt.show()
plt.close()

O gráfico mostra que na banda de passagem plana caracterísitca de filtros com janelamento butterworth. Além disso, há uma queda de -3dB na frequência de corte especificada $\omega_c = 0.65\pi$. Portanto, o gráfico está como esperado

8. Gere o gráfico da resposta em fase deste sistema (com fase contínua, usando unwrap. Não esqueça de colocar as frequências corretas no eixo x. Explique, em forma de comentário em seu código, se a fase da Resposta em Frequência está de acordo com o esperado.

In [ ]:
H_phase        = np.angle(H)
H_phase_unwrap = np.unwrap(H_phase)

In [ ]:
figure, axes = plt.subplots(nrows = 1, ncols = 2, figsize = (16, 6))

axes[0].plot(H_frequencies, H_phase)
axes[0].set_title("Fase de $ H(j\\omega) $")
axes[0].set_xlabel("Frequência (rad/amostra)")
axes[0].set_ylabel("Defasamento")

axes[1].plot(H_frequencies, H_phase_unwrap)
axes[1].set_title("Fase $ H(j\\omega) $ (Unwrap)")
axes[1].set_xlabel("Frequência (rad/amostra)")
axes[1].set_ylabel("Defasamento")

plt.show()
plt.close()

Diferentemente de filtros de fase linear, o filtro Butterworth é caracterizado por ter uma resposta de frequência maximamente plana na banda passante, mas isso é obtido ao custo de uma mudança de fase não-linear. Portanto, o gráfico está como esperado

9. Gere o gráfico do atraso de grupo deste sistema (pode usar a função pronta para o cálculo do atraso de grupo). Não esqueça de colocar as frequências corretas no eixo x. Explique, em forma de comentário em seu código, se o atraso de grupo está de acordo com o esperado.

Calculando o atraso de grupo numericamente

In [ ]:
d_H_phase_unwrap = np.diff(H_phase_unwrap)
d_H_frequencies  = np.diff(H_frequencies)

H_group_delay = - d_H_phase_unwrap / d_H_frequencies

In [ ]:
figure, axes = plt.subplots(nrows = 1, ncols = 2, figsize = (16, 6))

axes[0].plot(H_frequencies, H_phase_unwrap)
axes[0].set_title("Fase $ H(j\\omega) $ (Unwrap)")
axes[0].set_xlabel("Frequência (rad/amostra)")
axes[0].set_ylabel("Defasamento")

axes[1].plot(H_frequencies[1:], H_group_delay)
axes[1].set_title("Atraso de Grupo de $ H(j\\omega) $")
axes[1].set_xlabel("Frequência (rad/amostra)")
axes[1].set_ylabel("Defasamento")

plt.show()
plt.close()

Como a mudança da fase não é linear. Então, o atraso de grupo também não é. Portanto, o gráfico está como esperado

10. Filtre o sinal x[n] da Questão 1 usando os coeficientes ak e bk da equação de diferenças gerados na Questão 7. Gere o gráfico do módulo da Transformada de Fourier da saída y[n] em dB. Não esqueça de colocar as frequências corretas no eixo x. Explique, em forma de comentário em seu código, se o módulo da Transformada de Fourier do sinal filtrado está de acordo com o esperado.

Aplicando o filtro no sinal $x[n]$

In [ ]:
y = signal.lfilter(b_k, a_k, x)

Calculando a Transformada de Fourier de $ y[n] $

In [ ]:
N_y_fft = 2048

Y = np.fft.fft(y, N_y_fft)
Y = np.fft.fftshift(Y)

Y_abs    = np.abs(Y)
Y_abs_dB = 20 * np.log10(Y_abs)

Y_frequencies = np.linspace(-np.pi, np.pi, N_y_fft)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(x_n, y)
axes[0].set_title("$y[n]$")
axes[0].set_xlabel("n")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True)

axes[1].plot(Y_frequencies, Y_abs_dB)
axes[1].set_title("Módulo de $Y(j\\omega)$ (dB)")
axes[1].set_xlabel("Frequência (rad/amostra)")
axes[1].set_ylabel("Magnitude (dB)")
axes[1].axhline(y = -3, color = 'red', linestyle='--')
axes[1].grid(True)

plt.tight_layout()
plt.show()

O filtro Butterworth passa-baixa (fc = 0.65π) deve atenuar de forma suave a componente de maior frequência. Logo, no gráfico devem aparecer dois picosfortes em 0.2π e 0.5π. Portanto, o gráfico está como esperado

11. Novamente, o sinal de saída y[n] deve ser, de forma aproximada, igual a um sinal g[n] que corresponde à soma de dois cossenos com frequências angulares igual a 0,2π e 0,5π, mas com um atraso. Entretanto, desta vez, o atraso do sinal g[n] não pode ser determinado a priori. Este atraso deve ser estimado a partir do atraso de grupo. Para saber se isto realmente está acontecendo, gere, em um mesmo gráfico, os sinais g[n-nd] e y[n], em que nd é um atraso que você deve estimar a partir do atraso de grupo. Comente se estes sinais são parecidos e sincronizados.

Estimando o $n_d$ pela média dos atrasos das frequências $0,2\pi$ e $0,5\pi$

In [ ]:
idx_02 = np.argmin(np.abs(H_frequencies - 0.2 * np.pi))
idx_05 = np.argmin(np.abs(H_frequencies - 0.5 * np.pi))

nd = int((H_group_delay[idx_02] + H_group_delay[idx_05]) / 2)

print("nd =", nd, "amostras")

Gerando o sinal $ g[n] $

In [ ]:
g = np.cos(0.2 * np.pi * (x_n - nd)) + \
    np.cos(0.5 * np.pi * (x_n - nd))

Gerando o sinal $ e[n] = g[n] - y[n] $

In [ ]:
e = g - y

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey = True)

axes[0].plot(x_n, y, alpha = 0.75)
axes[0].plot(x_n, g, alpha = 0.75)
axes[0].set_title("$ g[n] $ e $ y[n] $")
axes[0].set_xlabel("n")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True)

axes[1].plot(x_n, e)
axes[1].set_title("$ e[n] = g[n] - y[n] $")
axes[1].set_xlabel("n")
axes[1].set_ylabel("Amplitude")
axes[1].grid(True)

plt.tight_layout()
plt.show()

Os gráficos são razoavelmente parecidos e sincronizados. Podemos notar isso no gráfico do sinal $e[n]$ que representa a diferença entre esses sinais